# Stream Processing Fundamentals: Micro-Batch vs Continuous Processing

**Module**: 09_streaming  
**Notebook**: 01_streaming_fundamentals.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The evolution of Spark streaming: legacy DStreams to Structured Streaming, the unbounded table mental model, and latency vs throughput trade-offs.
- **Why does this topic exist?**: Real-time streaming powers anomaly detection, live analytics dashboards, and event-driven architectures.
- **What problem does it solve?**: Managing unbounded streams without state store explosion.
- **Where is it used in real-world Data Engineering?**: Real-time payment fraud scoring and clickstream analysis.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrames, sources/sinks, basic architecture |
| **Required depth** | Working knowledge: can explain micro-batch execution and the unbounded-table model |
| **Data Engineer** | Build a basic streaming query and explain latency, state, and delivery semantics. |
| **Senior Data Engineer** | Choose micro-batch versus continuous processing for a workload. |
| **Spark specialist** | Reason about source/sink contracts and exactly-once requirements. |
| **Next topic** | `09_streaming/02_structured_streaming.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Structured Streaming treats a stream as an unbounded table that grows over time. Each micro-batch reads new
data, updates the result, and writes output. This model gives the same DataFrame API as batch processing, but the
guarantees depend on the source and sink: replayable input plus idempotent or transactional output is what makes
exactly-once possible.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Unbounded table** | Stream as a growing table | New rows are appended logically |
| **Micro-batch** | Periodic execution | Processes available data at each trigger |
| **Trigger** | When to run a batch | Default, processing time, availableNow, once, continuous |
| **Source** | Input contract | Must support replay for recovery |
| **Sink** | Output contract | Determines delivery guarantees |
| **Checkpoint** | Progress and state | Stores offsets, commits, and state |

### How it works, step by step

1. The query starts and reads the source's current offset.
2. A trigger fires and the engine reads new data since the last offset.
3. The batch is planned and executed like a batch DataFrame.
4. Results are written to the sink.
5. Offsets and state are committed to the checkpoint.
6. The next trigger repeats from the committed offset.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Structured Streaming Mental Model: Stream as an Unbounded Table
```
Incoming Event Stream:
[Event 1] ---> [Event 2] ---> [Event 3] ---> ... (Never ends!)
      |             |              |
      v             v              v
+-----------------------------------------------------------+
| UNBOUNDED INPUT TABLE (New rows appended continuously)    |
+-----------------------------------------------------------+
                            |
                            v Query Expression (DataFrame / SQL)
+-----------------------------------------------------------+
| RESULT TABLE (Updated incrementally per micro-batch)     |
+-----------------------------------------------------------+
                            |
                            v Output Mode (Append / Update / Complete)
+-----------------------------------------------------------+
| EXTERNAL SINK (Console, Files, Kafka, Delta Lake)        |
+-----------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Streaming_Fund_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Rate Stream Source Demo (Synthetic Stream)

In [ ]:
# Create built-in rate source (generates sequential numbers at specified rows/second)
rate_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

print("Is stream active in DataFrame?:", rate_stream.isStreaming)
rate_stream.printSchema()

---

## Experiment: Running a Short-Lived Streaming Query to In-Memory Sink

- **1. What are we testing?**: Start a micro-batch query writing to memory sink and inspect results.
- **2. What do we expect?**: Query populates an in-memory table that can be queried with spark.sql().

In [ ]:
query = (
    rate_stream
    .writeStream
    .format("memory")
    .queryName("rate_sink")
    .outputMode("append")
    .start()
)

# Wait 4 seconds for a few micro-batches to process
time.sleep(4)

print("Querying in-memory streaming sink:")
spark.sql("SELECT * FROM rate_sink ORDER BY timestamp DESC LIMIT 5").show()

query.stop()
print("Streaming query stopped successfully.")

**3. What happened?**: Rate stream generated timestamped integers into in-memory table.  
**4. Why did it happen?**: Structured Streaming micro-batches processed incoming rows as fast as possible.  
**5. Practical Engineering Lesson**: Memory sink is ideal for unit testing and interactive debugging in Jupyter.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Clickstream session metrics

- **Industry**: Digital media
- **Scenario**: A media company needs near-real-time session counts and top pages.
- **Problem**: Batch jobs are too slow for operational dashboards, but full real-time infrastructure is expensive.
- **Why this concept is relevant**: Micro-batch streaming provides low-minute latency with the same DataFrame API and checkpointed recovery.
- **Implementation idea**: Read events from a replayable source, aggregate per window, and write to a dashboard store.
- **Expected result**: Fresh metrics with bounded latency and recoverable progress.
- **Alternative**: A dedicated streaming engine with lower latency.
- **Trade-offs**: A dedicated engine may reduce latency but adds operational complexity and a second programming model.

### Use Case 2: Fraud alert enrichment

- **Industry**: Financial services
- **Scenario**: Transactions must be enriched with merchant risk scores within seconds.
- **Problem**: A batch pipeline misses the intervention window.
- **Why this concept is relevant**: Streaming enrichment with a broadcast lookup or stream-static join meets the latency requirement.
- **Implementation idea**: Read transactions, join a static risk table, and write alerts to a queue.
- **Expected result**: Alerts delivered within the operational window.
- **Alternative**: Score at the edge or in the application.
- **Trade-offs**: Edge scoring reduces latency but duplicates logic and complicates model updates.

---

## Business Value

- Lower latency for operational decisions.
- Reuse of batch code and skills.
- Recoverable progress through checkpoints.
- A clear path from batch to streaming.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Choose a replayable source for recoverability.
- Make the sink idempotent or transactional.
- Monitor batch duration, input rate, and state size.
- Use event-time processing when arrival order is unreliable.
- Do not assume exactly-once without checking the source and sink.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Observe micro-batch progress

**Steps**

1. Start a file-source stream with a trigger interval.
2. Add new files and watch batches and input rows in the query progress.
3. Stop and restart from the checkpoint and confirm it resumes without reprocessing committed data.

**What to observe**: Each trigger processes only new data and commits progress to the checkpoint.

**Healthy result**: Restart resumes from the last committed offset.

**Problematic result**: The query reprocesses data or fails because the checkpoint is missing/incompatible.

**Engineering takeaway**: The checkpoint is part of the query contract.

---

## Trade-offs and Alternatives

### Advantages

- Unified API
- Checkpointed recovery
- Supports event time and state

### Limitations

- Micro-batch latency
- State can grow
- Exactly-once depends on source/sink

### When to use it

- Dashboards
- Alerting
- Continuous ingestion

### When not to use it

- Sub-second latency with micro-batch
- Non-replayable sources without a plan
- Unbounded state

### Alternatives inside Spark

- Micro-batch engine
- Checkpoints
- Watermarks

### Alternatives outside Spark

- Flink
- Kafka Streams
- Managed streaming services

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Stream Status
**Problem Statement**: Query query.isActive and query.status.  
**Input / Context**: query

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Rate Source with Ramptime
**Problem Statement**: Configure rampUpTime option on rate source.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Explain Legacy DStreams
**Problem Statement**: Contrast DStreams with Structured Streaming.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Restart reprocessed everything

**Scenario**: A streaming job restarts and duplicates all output.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the checkpoint location was changed or deleted.
2. Verify the sink is idempotent.
3. Inspect the committed offsets and the source's replay behavior.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Stream Status):
```python
print('query.isActive, query.status')
```

#### Solution 2 (Rate Source with Ramptime):
```python
spark.readStream.format('rate').option('rampUpTime','5s').load()
```

#### Solution 3 (Explain Legacy DStreams):
```python
print('DStreams used RDDs; Structured Streaming uses the Catalyst DataFrame engine with native event-time.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Unbounded Table**: Stream is conceptually an infinite append-only table.
- **Micro-batch**: Processes incoming records in small discrete batches.

### Essential Technical Terms
- `isStreaming`: Boolean flag indicating streaming DataFrame.
- `rate source`: Synthetic benchmark source.

### Common Engineering Mistakes
- **Calling collect() on streaming DataFrame**: Throws AnalysisException; streaming data must use writeStream.

### Interview Questions on This Topic
- **Q: What is Structured Streaming?**
  - *A*: A scalable stream processing engine built on the Spark SQL engine, treating streams as unbounded tables.


### More Interview Questions

- **Q: What is the unbounded-table model?**
  - *A*: A stream is treated as a table that grows over time; each query incrementally updates its result as new rows arrive.
- **Q: What determines exactly-once in Structured Streaming?**
  - *A*: A replayable source plus a sink that is idempotent or transactional, together with a durable checkpoint that records offsets and commits.

### What to Study Next
Open 02_structured_streaming.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')